# REPSOL Model Training (70/15/15)

**EfficientNet-03 — Targeted improvements over EfficientNet-02**

EfficientNet-02 achieved 70% test accuracy but had clear weaknesses:
- **Class 4 (SPL BAJO):** F1=0.46 — 21/39 samples misclassified as class 3 (SPL ALTO). Both are continuous tones at the same frequency; only amplitude differs, which z-score normalisation wipes out.
- **Class 1 (Pulses HAMMERING):** F1=0.36 — only 6 test samples, model rarely predicts this class.
- **Class 7 (Works/sirens):** F1=0.22 — only 6 test samples, worst performing class.

**Changes in this run vs EfficientNet-02:**

| What | EfficientNet-02 | EfficientNet-03 | Why |
|------|----------------|----------------|-----|
| LR | 1e-3 | 5e-4 | Slower start, less aggressive early updates |
| LR schedule | ReduceLROnPlateau | OneCycleLR (warmup + cosine) | Smoother decay, avoids plateau traps |
| Loss | Weighted CrossEntropy | Weighted CrossEntropy + label smoothing 0.1 | Prevents overconfidence on dominant classes |
| Class weights | sklearn balanced | sklearn balanced + ×1.5 boost for classes 1 & 7 | Mild extra push on the two worst classes without destabilising training |
| Patience | 4 | 6 | More time to escape local minima |
| Epochs | 15 | 25 | More budget given cosine schedule |
| Batch size | 8 | 8 | Unchanged |
| Backbone | EfficientNet-B0 | EfficientNet-B0 | Same pretrained weights |

> **Why not squared weights?** A 17x class imbalance squared gives a ~300x loss weight ratio. That means the model effectively ignores 61% of training data (classes 3 and 6). Training collapses. Sklearn balanced weights already correct for imbalance correctly; a small manual boost on top is safer.

## 0. Config

In [1]:
from pathlib import Path
import sys
import torch

# ===== Hyperparameters =====
BATCH_SIZE    = 8
EPOCHS        = 25
LEARNING_RATE = 5e-4
PATIENCE      = 6
LABEL_SMOOTHING = 0.1
MODEL_NAME    = "efficientnet"

# ===== Paths =====
PROJECT_ROOT    = Path(r"D:\Work\Internships\INMAR\REPSOL")
SPECTROGRAM_DIR = PROJECT_ROOT / "Data" / "Spectrograms"
OUTPUT_DIR      = PROJECT_ROOT / "Models_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))


def next_run_path(model_name, suffix, ext, output_dir):
    prefix = f"{model_name}{suffix}"
    existing = [0]
    for p in output_dir.iterdir():
        if not p.is_file() or p.suffix != ext:
            continue
        stem = p.stem
        if stem.startswith(prefix + "_"):
            tail = stem[len(prefix) + 1:]
            if tail.isdigit():
                existing.append(int(tail))
    return output_dir / f"{prefix}_{max(existing)+1:02d}{ext}"


CHECKPOINT_PATH = next_run_path(MODEL_NAME, "_best", ".pth", OUTPUT_DIR)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("CHECKPOINT_PATH:", CHECKPOINT_PATH)
print("DEVICE         :", DEVICE)
print(f"LR={LEARNING_RATE}  EPOCHS={EPOCHS}  PATIENCE={PATIENCE}  LABEL_SMOOTHING={LABEL_SMOOTHING}")

CHECKPOINT_PATH: D:\Work\Internships\INMAR\REPSOL\Models_output\efficientnet_best_03.pth
DEVICE         : cpu
LR=0.0005  EPOCHS=25  PATIENCE=6  LABEL_SMOOTHING=0.1


## 1. Verify Data

In [2]:
def count_pt_files(root):
    counts = {}
    for split in ["train", "val", "test"]:
        split_dir = root / split
        counts[split] = sum(1 for _ in split_dir.rglob("*.norm.pt")) if split_dir.exists() else 0
    return counts

counts = count_pt_files(SPECTROGRAM_DIR)
print("Tensor files by split:", counts)
print("Total:", sum(counts.values()))

assert counts["train"] > 0, "No train .norm.pt files found."
assert counts["val"]   > 0, "No val .norm.pt files found."
assert counts["test"]  > 0, "No test .norm.pt files found."

Tensor files by split: {'train': 1382, 'val': 296, 'test': 297}
Total: 1975


## 2. Check per-class distribution

In [3]:
import os
import pandas as pd

train_dir = SPECTROGRAM_DIR / "train"
class_counts = {
    cls: sum(1 for _ in (train_dir / cls).rglob("*.norm.pt"))
    for cls in sorted(os.listdir(train_dir))
    if (train_dir / cls).is_dir()
}

df_dist = pd.DataFrame.from_dict(class_counts, orient="index", columns=["train_count"])
df_dist["short"] = df_dist.index.str[:40]
print("Training set class distribution:")
print(df_dist[["short", "train_count"]].to_string(index=False))
print(f"\nMin: {df_dist.train_count.min()}  Max: {df_dist.train_count.max()}  Imbalance ratio: {df_dist.train_count.max()/df_dist.train_count.min():.1f}x")

Training set class distribution:
                                   short  train_count
     0 ActividadBASE_NO pattern activity          158
                      1 Pulses HAMMERING           30
              2 Marked cycles 3 segundos           92
3 Continuous activity & tone 3.15kHz_SPL          492
4 Continuous activity & tone 3.15kHz_ SP          178
                                5 Blasts           51
         6 Machinery continuous activity          352
7 Works_ sirens and knocks en altas frec           29

Min: 29  Max: 492  Imbalance ratio: 17.0x


## 3. Dependencies

In [4]:
import importlib, subprocess, sys

required = ["torch", "torchvision", "torchaudio", "scikit-learn", "pandas", "tqdm", "numpy"]
name_map = {"scikit-learn": "sklearn"}
for pkg in required:
    try:
        importlib.import_module(name_map.get(pkg, pkg.replace("-", "_")))
        print(f"OK: {pkg}")
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        print(f"Installed: {pkg}")

OK: torch
OK: torchvision
OK: torchaudio
OK: scikit-learn
OK: pandas
OK: tqdm
OK: numpy


## 4. Training

In [5]:
import importlib
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR
from pathlib import Path
from tqdm import tqdm

import src.EfficientNet.model as model_module
from src.dataloaders import get_dataloaders

model_module = importlib.reload(model_module)
EfficientNetSpectrogram = model_module.EfficientNetSpectrogram
compute_class_weights   = model_module.compute_class_weights

# ── DataLoaders ──
train_loader, val_loader, test_loader = get_dataloaders(
    SPECTROGRAM_DIR, batch_size=BATCH_SIZE,
    num_workers=0, pin_memory=False, persistent_workers=False,
)
NUM_CLASSES = len(train_loader.dataset.classes)
CLASS_NAMES = train_loader.dataset.classes
print("Classes:", NUM_CLASSES, "  Train batches:", len(train_loader))

# ── Model ──
model = EfficientNetSpectrogram(num_classes=NUM_CLASSES, freeze_backbone=False).to(DEVICE)

# ── Class weights: sklearn balanced, then ×1.5 boost for classes 1 and 7 ──
# sklearn balanced already corrects the 17x imbalance.
# A small manual multiplier on top nudges the two worst classes
# without distorting the loss landscape.
BOOST_CLASSES  = {1: 1.5, 7: 1.5}   # index → multiplier

train_labels  = [label for (_, label) in train_loader.dataset.samples]
base_weights  = compute_class_weights(train_labels, num_classes=NUM_CLASSES).numpy()
final_weights = base_weights.copy()
for cls_idx, multiplier in BOOST_CLASSES.items():
    final_weights[cls_idx] *= multiplier
class_weights = torch.tensor(final_weights, dtype=torch.float).to(DEVICE)

print("\nClass weights:")
for i, (name, bw, fw) in enumerate(zip(CLASS_NAMES, base_weights, final_weights)):
    boost = f" ×{BOOST_CLASSES[i]}" if i in BOOST_CLASSES else ""
    print(f"  [{i}] {name[:45]:45s}  balanced={bw:.3f}  final={fw:.3f}{boost}")

# ── Loss: weighted CrossEntropy + label smoothing ──
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)

# ── Optimiser ──
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

# ── Scheduler: OneCycleLR (linear warmup → cosine decay) ──
total_steps = EPOCHS * len(train_loader)
scheduler = OneCycleLR(
    optimizer,
    max_lr=LEARNING_RATE,
    total_steps=total_steps,
    pct_start=0.1,        # 10% of steps = warmup
    anneal_strategy="cos",
    div_factor=10.0,      # start LR = max_lr / 10
    final_div_factor=100, # end LR = start_lr / 100
)

# ── Checkpoint + history setup ──
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
HISTORY_PATH = CHECKPOINT_PATH.with_name(f"{CHECKPOINT_PATH.stem}_training_history.csv")
for p in (CHECKPOINT_PATH, HISTORY_PATH):
    if p.exists(): p.unlink()
with open(HISTORY_PATH, "w") as fh:
    fh.write("epoch,train_loss,val_loss,train_acc,val_acc,lr\n")
torch.save(model.state_dict(), CHECKPOINT_PATH)

# ── Training loop ──
best_val_loss = float("inf")
no_improve = 0

for epoch in range(1, EPOCHS + 1):

    # --- train ---
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS} Train", ncols=100, unit="batch")
    for x, y in pbar:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        t_loss   += loss.item()
        t_correct += (out.argmax(1) == y).sum().item()
        t_total   += y.size(0)
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
    train_loss = t_loss / len(train_loader)
    train_acc  = 100.0 * t_correct / t_total

    # --- validate ---
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    pbar = tqdm(val_loader, desc=f"Epoch {epoch:02d}/{EPOCHS} Val  ", ncols=100, unit="batch")
    with torch.no_grad():
        for x, y in pbar:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out  = model(x)
            loss = criterion(out, y)
            v_loss   += loss.item()
            v_correct += (out.argmax(1) == y).sum().item()
            v_total   += y.size(0)
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
    val_loss = v_loss / len(val_loader)
    val_acc  = 100.0 * v_correct / v_total

    current_lr = optimizer.param_groups[0]["lr"]
    print(
        f"Epoch {epoch:02d}/{EPOCHS} "
        f"| Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} "
        f"| Train Acc: {train_acc:.2f} | Val Acc: {val_acc:.2f} "
        f"| LR: {current_lr:.2e}",
        flush=True,
    )

    with open(HISTORY_PATH, "a") as fh:
        fh.write(f"{epoch},{train_loss:.6f},{val_loss:.6f},{train_acc:.4f},{val_acc:.4f},{current_lr:.6f}\n")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        no_improve = 0
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f"  ✓ Saved improved checkpoint → {CHECKPOINT_PATH.name}", flush=True)
    else:
        no_improve += 1
        print(f"  No improvement {no_improve}/{PATIENCE}", flush=True)
        if no_improve >= PATIENCE:
            print(f"  Early stopping.", flush=True)
            break

print("\nTraining finished.")
print(f"Best val loss: {best_val_loss:.4f}")
print(f"Checkpoint: {CHECKPOINT_PATH}")

Classes: 8   Train batches: 173

Class weights:
  [0] 0 ActividadBASE_NO pattern activity            balanced=1.093  final=1.093
  [1] 1 Pulses HAMMERING                             balanced=5.758  final=8.637 ×1.5
  [2] 2 Marked cycles 3 segundos                     balanced=1.878  final=1.878
  [3] 3 Continuous activity & tone 3.15kHz_SPL ALTO  balanced=0.351  final=0.351
  [4] 4 Continuous activity & tone 3.15kHz_ SPL BAJ  balanced=0.971  final=0.971
  [5] 5 Blasts                                       balanced=3.387  final=3.387
  [6] 6 Machinery continuous activity                balanced=0.491  final=0.491
  [7] 7 Works_ sirens and knocks en altas frecuenci  balanced=5.957  final=8.935 ×1.5


Epoch 01/25 Val  : 100%|████████████████████████████| 37/37 [05:31<00:00,  8.97s/batch, loss=1.8053]

Epoch 01/25 | Train Loss: 2.4101 | Val Loss: 2.7373 | Train Acc: 17.66 | Val Acc: 37.16 | LR: 2.06e-04


  ✓ Saved improved checkpoint → efficientnet_best_03.pth


Epoch 02/25 Val  : 100%|████████████████████████████| 37/37 [05:25<00:00,  8.81s/batch, loss=1.6248]

...

Epoch 16/25 | Train Loss: 1.2117 | Val Loss: 2.4048 | Train Acc: 98.84 | Val Acc: 68.92 | LR: 1.73e-04
  No improvement 3/6



Epoch 17/25 Val  : 100%|████████████████████████████| 37/37 [05:27<00:00,  8.84s/batch, loss=0.8176]

Epoch 17/25 | Train Loss: 1.2069 | Val Loss: 2.4179 | Train Acc: 99.35 | Val Acc: 67.23 | LR: 1.41e-04
  No improvement 4/6



Epoch 18/25 Train:  58%|███████████████           | 100/173 [28:48<21:01, 17.28s/batch, loss=1.9124]


KeyboardInterrupt: 

## 5. Evaluation

In [ ]:
import importlib
import src.evaluate as eval_module

eval_module = importlib.reload(eval_module)
evaluate_model = eval_module.evaluate_model

# Load best checkpoint
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))

val_metrics  = evaluate_model(model, val_loader,  DEVICE)
test_metrics = evaluate_model(model, test_loader, DEVICE)

print("Validation:")
print({k: round(val_metrics[k], 4) for k in ["accuracy", "precision", "recall", "f1"]})
print("\nTest:")
print({k: round(test_metrics[k], 4) for k in ["accuracy", "precision", "recall", "f1"]})

In [ ]:
print("Test Classification Report:\n")
print(test_metrics["report"])
print("Confusion Matrix:")
print(test_metrics["confusion_matrix"])

## 6. Learning Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(HISTORY_PATH)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(df["epoch"], df["train_acc"], label="train")
axes[0].plot(df["epoch"], df["val_acc"],   label="val", linestyle="--")
axes[0].set_title("Accuracy"); axes[0].set_xlabel("Epoch"); axes[0].legend()

axes[1].plot(df["epoch"], df["train_loss"], label="train")
axes[1].plot(df["epoch"], df["val_loss"],   label="val", linestyle="--")
axes[1].set_title("Loss"); axes[1].set_xlabel("Epoch"); axes[1].legend()

axes[2].plot(df["epoch"], df["lr"])
axes[2].set_title("Learning Rate (OneCycleLR)"); axes[2].set_xlabel("Epoch")
axes[2].set_yscale("log")

fig.suptitle("EfficientNet-03 Learning Curves", fontsize=13)
fig.tight_layout()
plt.savefig(PROJECT_ROOT / "outputs" / "evaluation" / "learning_curves_efficientnet_03.png",
            dpi=150, bbox_inches="tight")
plt.show()